In [59]:
import os
import uuid
from datetime import datetime
import boto3

# Generate a new job id
job_id = str(uuid.uuid4())
input_bucket_name = "loc-preservation"#os.getenv('INPUT_BUCKET_NAME')
output_bucket_name = "ndnp-open-ocr-output-bucket-test"#os.getenv('OUTPUT_BUCKET_NAME')

os.environ['AWS_PROFILE'] = "NDNP_OPEN_OCR_DEVELOPER_DEV_profile"
print(os.environ.get('AWS_PROFILE'))

input_prefix = "loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/"#event['pathParameters']['prefix']
job_id = '354ad159-b8ba-47ba-a7a5-1de3f28b41c2'
output_prefix = "batch_dlc_alice_ver01__{}".format(job_id)

if input_bucket_name is None or output_bucket_name is None:
    raise Exception("Missing S3_BUCKET_NAME environment variables")

session = boto3.Session(profile_name=os.environ['AWS_PROFILE'])
s3 = session.resource("s3")
sqs = boto3.client('sqs')
dynamodb = boto3.resource('dynamodb')
# table = dynamodb.Table(os.getenv('TABLE_NAME'))

input_keys = set()
output_keys = set()

# Collect keys from the input bucket
for object_summary in s3.Bucket(input_bucket_name).objects.filter(Prefix=input_prefix):
    if object_summary.key.lower().endswith('.pdf'):
        input_keys.add(object_summary.key)

# Collect keys from the output bucket
for object_summary in s3.Bucket(output_bucket_name).objects.filter(Prefix=output_prefix):
    if object_summary.key.lower().endswith('.pdf'):
        output_keys.add(object_summary.key)

# Find missing keys
missing_keys = input_keys - output_keys

print(missing_keys)
print(str(len(missing_keys)) + "MISSING PDFS total in the batch")
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")
# output_prefix = os.path.split(prefix)[1] + "__" + job_id
messages = []

# # Update DynamoDB table
# table.put_item(
#     Item={
#         'pk': 'JOB',
#         'sk': job_id,
#         'TotalMessages': len(missing_keys),
#         'RemainingMessages': len(missing_keys),
#         'Timestamp': timestamp
#     }
# )

queue_url = os.getenv('QUEUE_URL')
alto_queue_url = os.getenv('ALTO_QUEUE_URL')


NDNP_OPEN_OCR_DEVELOPER_DEV_profile
{'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530698/1861123001/0851.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530698/1861102201/0382.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530686/1861082801/0945.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860051001/0078.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530662/1860090801/0059.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530698/1861101801/0356.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530686/1861061101/0327.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860080701/0702.pdf', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860050201/0022.pdf', 'loc-preservation/lcbp/ndnp

In [60]:
# Take the filename of the keys (last path component and find which are not included in both)
cleaned_inputs = set()
cleaned_outputs = set()
for key in input_keys:
    cleaned_inputs.add(key.split("data", 1)[-1] )
for key in output_keys:
    cleaned_outputs.add(key.split("data", 1)[-1] )
key = "/path/to/key"


In [61]:
print(cleaned_outputs)

{'/sn83030213/00206530686/1861080401/0752.pdf', '/sn83030213/00206530698/1861120401/0674.pdf', '/sn83030213/00206530650/1860080901/0718.pdf', '/sn83030213/00206530674/1861040101/0621.pdf', '/sn83030213/00206530662/1860091201/0084.pdf', '/sn83030213/00206530686/1861062301/0426.pdf', '/sn83030213/00206530686/1861061201/0341.pdf', '/sn83030213/00206530650/1860081601/0767.pdf', '/sn83030213/00206530686/1861070701/0533.pdf', '/sn83030213/00206530649/1860041801/0769.pdf', '/sn83030213/00206530650/1860060801/0292.pdf', '/sn83030213/00206530686/1861052801/0229.pdf', '/sn83030213/00206530649/1860032001/0558.pdf', '/sn83030213/00206530698/1861112901/0639.pdf', '/sn83030213/00206530698/1861112501/0614.pdf', '/sn83030213/00206530674/1861033001/0612.pdf', '/sn83030213/00206530649/1860041601/0754.pdf', '/sn83030213/00206530662/1860111501/0533.pdf', '/sn83030213/00206530686/1861080401/0751.pdf', '/sn83030213/00206530662/1860121801/0775.pdf', '/sn83030213/00206530698/1861090701/0061.pdf', '/sn83030213

In [62]:
missing_keys = cleaned_inputs - cleaned_outputs
print(len(missing_keys))

0


In [63]:
print(len(cleaned_inputs))

5276


In [64]:
print(len(input_keys))

5276


In [65]:
print(missing_keys)

set()


In [66]:
# Create a new set containing only the elements that start with one of the prefixes
filtered_elements = {element.replace('.pdf', '.tif') for element in input_keys if any(element.endswith(prefix) for prefix in missing_keys)}

In [67]:
filtered_elements

set()

In [68]:
print(len(filtered_elements))

0


In [58]:
import uuid
import boto3
import json
sqs = session.client('sqs')
# Retry a batch of 10 to see how they perform with the new preprocessing settings... Sample SQS message 
os.environ['AWS_PROFILE'] = "NDNP_OPEN_OCR_DEVELOPER_DEV_profile"
print(os.environ['AWS_PROFILE'])
"""
{'InputPrefix': 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01', 'Bucket': 'loc-preservation', 'Key': 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860082701/0836.tif', 'OutputPrefix': 'batch_dlc_alice_ver01__354ad159-b8ba-47ba-a7a5-1de3f28b41c2', 'JobId': '354ad159-b8ba-47ba-a7a5-1de3f28b41c2'}
"""
msgs = []
for element in list(filtered_elements)[10:]:
    msg = {
        'InputPrefix': input_prefix,
        'Bucket': input_bucket_name,
        'Key': element,
        'OutputPrefix': output_prefix,
        'JobId': job_id
    }

    sqs_msg = {
        'Id': str(uuid.uuid4()),
        'MessageBody': json.dumps(msg)
    }

    sqs.send_message(QueueUrl="https://sqs.us-east-2.amazonaws.com/342134162356/ndnp-open-ocr-pdf-consumer-sqs-queue", MessageBody=json.dumps(msg))
    

NDNP_OPEN_OCR_DEVELOPER_DEV_profile


In [47]:
print(filtered_elements)

{'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530674/1861032201/0562.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860080301/0677.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530674/1861030201/0420.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530698/1861101701/0345.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530686/1861050801/0064.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530686/1861080901/0792.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530698/1861091501/0119.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530650/1860082701/0836.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83030213/00206530686/1861070201/0496.tif', 'loc-preservation/lcbp/ndnp/dlc/batch_dlc_alice_ver01/data/sn83

In [49]:
with open('pdf.txt','w') as f:
   f.write(str(filtered_elements))  # set of numbers & a tuple

In [1]:
    # Initialize DynamoDB resource
    dynamodb = boto3.resource("dynamodb")

    # DynamoDB table name
    table_name = os.getenv("TABLE_NAME")

    # Initialize table resource
    table = dynamodb.Table(table_name)

    # Use the primary key to fetch the item
    # In this example, 'ItemID' is the name of the primary key (it could be different in your table)
    # and '1' is the value of the primary key you are looking for.
    response = table.get_item(Key={"pk": "JOB", "sk": event["pathParameters"]["JobId"]})

    # Fetch item from the response
    item = response.get("Item", {})

    # Print the item
    print(item)


NameError: name 'boto3' is not defined